In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "tensor_engine").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "tensor_engine").is_dir():
    raise RuntimeError("Abre el notebook desde TensorEngine o TensorEngine/notebooks.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tensor_engine import (
    DimensionSpec, LagrangianSourceSpec, ParameterSpec, DisplayPolicy,
    TensorEngine, WolframXActBridge, draft4_circular_ansatz,
    spatially_flat_flrw_ansatz, expr_to_latex,
)

In [2]:
ANSATZ = "draft4"  # Caso-2 original: 3D, phi=p*varphi.
# La opción flrw conserva la misma fórmula en 4D; no es el caso original.

if ANSATZ == "draft4":
    ansatz = draft4_circular_ansatz()
    dimension = DimensionSpec(3)
elif ANSATZ == "flrw":
    ansatz = spatially_flat_flrw_ansatz()
    dimension = DimensionSpec(4)
else:
    raise ValueError(f"Ansatz desconocido: {ANSATZ}")



source = LagrangianSourceSpec(
    name="eqt_case2_" + ANSATZ,
    expression="R + 2/ell**2 - alpha_1*X + ell**2*beta0*(3*RicciUU - X*R)",
    dimension=dimension,
    parameters=(
        ParameterSpec("alpha_1", description="Constante cinética alpha_1; no nula"),
        ParameterSpec("ell", description="Escala L del artículo; no nula"),
        ParameterSpec("beta0", description="Acoplamiento beta_0 no nulo"),
        ParameterSpec("p", description="Carga del ansatz phi=p*varphi"),
    ),
    assumptions=("ell != 0", "beta0 != 0"),
    metadata=(("case", "EQT Caso-n: R + 2/ell^2 - alpha_1*X + ell^2 beta0 (3 Ricci_uu - X R)"),),
)
model = source.compile()
# Normalización 1: omitimos 1/(16*pi*G), constante global, en esta prueba.
# El motor ya trata la variación de sqrt(|g|); no incluirla en expression.
print("Modelo:", model.name, "| Ansatz:", ansatz.name, "| Dimensión:", dimension.value)

Modelo: eqt_case2_draft4 | Ansatz: draft4_circular | Dimensión: 3


In [3]:
VALIDAR_XACT = True  # False: solo verificaciones Python, sin afirmar validación xAct.

# Misma política de presentación utilizada en los reportes revisados.
# Solo afecta LaTeX/PDF; no modifica la IR canónica ni la validación.
display_policy = DisplayPolicy(
    factor=True,
    collect=True,
    together=True,
    canonicalize_indices=True,
    aggressive=False,
    enabled=True,
    max_nodes=4000,
)

def mostrar_progreso(event):
    print(f"{event.stage_key}: {event.state} ({event.duration_seconds:.1f} s)", flush=True)

run = TensorEngine(event_handler=mostrar_progreso).run(
    model,
    ansatz=ansatz,
    output_root=PROJECT_ROOT / "outputs" / "notebook_quickstart",
    display_policy=display_policy,
    wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None,
)
print("Estado:", run.status.value)
print("Verificaciones:", run.package.verification.summary)
print("Run ID:", run.package.run_id)

validate_model: started (0.0 s)
validate_model: completed (0.0 s)
normalize_lagrangian: started (0.0 s)
normalize_lagrangian: completed (0.0 s)


derive_momenta: started (0.0 s)
derive_momenta: completed (8.1 s)
raw_variation: started (0.0 s)
raw_variation: completed (0.1 s)
integrate_by_parts: started (0.0 s)
integrate_by_parts: completed (3.3 s)
noether: started (0.0 s)
noether: completed (6.6 s)
components: started (0.0 s)
components: completed (8.3 s)
wolfram_model_validation: started (0.0 s)
wolfram_model_validation: completed (26.5 s)
verify: started (0.0 s)
verify: completed (19.9 s)
derive_intermediate_quantities: started (0.0 s)
derive_intermediate_quantities: completed (20.3 s)
organize_result_views: started (0.0 s)
organize_result_views: completed (3.0 s)
export: started (0.0 s)
export: completed (11.0 s)
Estado: partial
Verificaciones: {'passed': 46, 'failed': 0, 'undetermined': 14}
Run ID: run_04cb0bb1806b83c04954


In [4]:
from IPython.display import FileLink, Markdown, display
import os

display(Markdown(
    "### EQT: Caso-n\n\n"
    + f"- Estado: **{run.status.value}**; verificaciones: `{run.package.verification.summary}`\n"
    + f"- Ansatz: `{ansatz.name}` ({ansatz.dimension}D)\n"
    + f"- $L = {expr_to_latex(run.abstract.lagrangian)}$\n"
    + f"- $F_\\phi = {expr_to_latex(run.abstract.scalar_derivative)}$\n"
))
if run.projected.lagrangian.components is not None:
    display(Markdown(rf"$$L_{{\mathrm{{ansatz}}}} = {expr_to_latex(run.projected.lagrangian.scalar)}$$"))
rows = ["| Cantidad | Proyección | Motivo |", "|---|---|---|"]
rows += [f"| `{q.key}` | {q.status.value} | {q.reason} |" for q in run.projected.quantities]
display(Markdown("\n".join(rows)))
for check in run.package.verification.checks:
    if check.status.value != "passed":
        print("Revisar:", check.key, check.status.value, check.message[:280])
if run.export_bundle is not None:
    folder = run.export_bundle.output_directory
    print("Bundle:", folder)
    for filename in ("report.pdf", "report.tex", "results.json", "manifest.json", "presentation.json"):
        path = folder / filename
        if path.exists():
            display(FileLink(os.path.relpath(path, Path.cwd())))
    if run.export_bundle.pdf_diagnostic:
        print("PDF:", run.export_bundle.pdf_diagnostic)
# Todas las expresiones: run.abstract y run.projected; E_ab: run.abstract.metric_euler.

### EQT: Caso-n

- Estado: **partial**; verificaciones: `{'passed': 46, 'failed': 0, 'undetermined': 14}`
- Ansatz: `draft4_circular` (3D)
- $L = -1\,{\mathrm{ell}}^{2}\,u{}_{d0}\,g{}^{d0}{}^{d1}\,u{}_{d1}\,R{}_{d2}{}_{d3}{}_{d4}{}_{d5}\,g{}^{d2}{}^{d4}\,g{}^{d3}{}^{d5}\,\mathrm{beta0} - 1\,u{}_{d0}\,g{}^{d0}{}^{d1}\,u{}_{d1}\,\mathrm{alpha}_{\mathrm{1}} + 2\,{{\mathrm{ell}}^{2}}^{-1} + 3\,{\mathrm{ell}}^{2}\,R{}_{d0}{}_{d1}{}_{d2}{}_{d3}\,g{}^{d0}{}^{d2}\,g{}^{d1}{}^{d4}\,g{}^{d3}{}^{d5}\,u{}_{d4}\,u{}_{d5}\,\mathrm{beta0} + R{}_{d0}{}_{d1}{}_{d2}{}_{d3}\,g{}^{d0}{}^{d2}\,g{}^{d1}{}^{d3}$
- $F_\phi = 0$


$$L_{\mathrm{ansatz}} = -1\,f^{(2)}\!\left(r\right) + 2\,{\mathrm{ell}}^{-2} - 2\,{r}^{-1}\,f^{(1)}\!\left(r\right) - 1\,\mathrm{alpha}_{\mathrm{1}}\,{p}^{2}\,{r}^{-2} + \mathrm{beta0}\,{\mathrm{ell}}^{2}\,{p}^{2}\,{r}^{-2}\,f^{(2)}\!\left(r\right) - 1\,\mathrm{beta0}\,{\mathrm{ell}}^{2}\,{p}^{2}\,{r}^{-3}\,f^{(1)}\!\left(r\right)$$

| Cantidad | Proyección | Motivo |
|---|---|---|
| `lagrangian` | completed | Proyectada respetando el ansatz 'draft4_circular'. |
| `metric_momentum` | completed | Proyectada respetando el ansatz 'draft4_circular'. |
| `curvature_momentum` | completed | Proyectada respetando el ansatz 'draft4_circular'. |
| `scalar_gradient_momentum` | completed | Proyectada respetando el ansatz 'draft4_circular'. |
| `scalar_derivative` | completed | La expresión abstracta es nula; todas sus componentes son cero. |
| `metric_euler` | completed | Reutilizada desde la etapa de componentes de E_ab. |
| `scalar_euler` | completed | Reutilizada desde la etapa de componentes de E_phi. |
| `ricci_scalar` | completed | Proyectada respetando el ansatz 'draft4_circular'. xAct se ejecutó, pero esta cantidad no tiene una identidad independiente. |
| `riemann_tensor` | completed | Proyectada respetando el ansatz 'draft4_circular'. xAct se ejecutó, pero esta cantidad no tiene una identidad independiente. |
| `nabla_P` | completed | Proyectada respetando el ansatz 'draft4_circular'. xAct se ejecutó, pero esta cantidad no tiene una identidad independiente. |
| `nabla_nabla_P` | completed | Proyectada respetando el ansatz 'draft4_circular'. xAct se ejecutó, pero esta cantidad no tiene una identidad independiente. |

Revisar: noether_current_decomposition undetermined La descomposición fue construida, pero su reducción completa requiere identidades multitémino de un backend externo.
Revisar: diffeomorphism_noether_identity undetermined La identidad requiere Bianchi diferencial o conmutación de derivadas que el backend estructural no fuerza.
Revisar: external.model.metric_momentum_symmetry undetermined M_ab es simétrico para este modelo. El residual IR no pudo transportarse. Residual xAct: IR decode failed
Revisar: external.model.curvature_momentum_first_pair undetermined P^{abcd} es antisimétrico en el primer par. El residual IR no pudo transportarse. Residual xAct: IR decode failed
Revisar: external.model.curvature_momentum_second_pair undetermined P^{abcd} es antisimétrico en el segundo par. El residual IR no pudo transportarse. Residual xAct: IR decode failed
Revisar: external.model.curvature_momentum_pair_exchange undetermined P^{abcd}=P^{cdab}. El residual IR no pudo transportarse. Residual xA

c:\Investigacion\TensorEngine\outputs\notebook_quickstart\eqt-case2-draft4-04cb0bb1806b\report.tex

c:\Investigacion\TensorEngine\outputs\notebook_quickstart\eqt-case2-draft4-04cb0bb1806b\results.json

c:\Investigacion\TensorEngine\outputs\notebook_quickstart\eqt-case2-draft4-04cb0bb1806b\manifest.json

c:\Investigacion\TensorEngine\outputs\notebook_quickstart\eqt-case2-draft4-04cb0bb1806b\presentation.json

PDF: La compilación LaTeX no produjo report.pdf. x.sty)
) (D:\MiKTeX\tex/latex/geometry\geometry.cfg))
(D:\MiKTeX\tex/latex/amsmath\amsmath.sty
For additional information on amsmath, use the `?' option.
(D:\MiKTeX\tex/latex/amsmath\amstext.sty
(D:\MiKTeX\tex/latex/amsmath\amsgen.sty))
(D:\MiKTeX\tex/latex/amsmath\amsbsy.sty)
(D:\MiKTeX\tex/latex/amsmath\amsopn.sty))
(D:\MiKTeX\tex/latex/amsfonts\amssymb.sty
(D:\MiKTeX\tex/latex/amsfonts\amsfonts.sty))
(D:\MiKTeX\tex/latex/breqn\breqn.sty (D:\MiKTeX\tex/latex/l3kernel\expl3.sty
(D:\MiKTeX\tex/latex/l3backend\l3backend-pdftex.def))
(D:\MiKTeX\tex/latex/graphics\graphicx.sty
(D:\MiKTeX\tex/latex/graphics\graphics.sty
(D:\MiKTeX\tex/latex/graphics\trig.sty)
(D:\MiKTeX\tex/latex/graphics-cfg\graphics.cfg)
(D:\MiKTeX\tex/latex/graphics-def\pdftex.def)))
(D:\MiKTeX\tex/latex/breqn\flexisym.sty (D:\MiKTeX\tex/latex/breqn\cmbase.sym)
(D:\MiKTeX\tex/latex/breqn\mathstyle.sty)) (D:\MiKTeX\tex/latex/tools\calc.sty)
) (D:\MiKTeX\tex/latex/needspace